In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

2026-03-26 16:35:47.138798: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-26 16:35:47.449936: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
digits = load_digits()
X = digits.data
y = digits.target

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (1797, 64)
Shape of y: (1797,)


In [ ]:
X = X / 16.0
y = to_categorical(y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
def create_model(hidden_layers):
    model = Sequential()

    # Input layer
    model.add(Dense(64, activation='relu', input_shape=(64,)))

    # Hidden layers
    for neurons in hidden_layers:
        model.add(Dense(neurons, activation='relu'))

    # Output layer
    model.add(Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [5]:
configs = {
    "1 Layer": [64],
    "2 Layers": [64, 32],
    "3 Layers": [64, 32, 16]
}

histories = {}
accuracies = {}

In [6]:
for name, layers in configs.items():
    print(f"\nTraining model: {name}")

    model = create_model(layers)

    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)

    histories[name] = history
    accuracies[name] = acc

    print(f"{name} Test Accuracy: {acc:.4f}")


Training model: 1 Layer


/home/smit/Documents/tf-env/lib64/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1774543022.257000   13732 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4623 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-03-26 16:37:03.832949: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fb04c017870 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-26 16:37:03.832974: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-26 16:37:03.865138: I tensorflow/compil

1 Layer Test Accuracy: 0.9583

Training model: 2 Layers
2 Layers Test Accuracy: 0.9667

Training model: 3 Layers
3 Layers Test Accuracy: 0.9583


In [7]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

names  = list(accuracies.keys())
values = list(accuracies.values())

# ── Global style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.spines.left':   False,
    'axes.spines.bottom': False,
    'axes.facecolor':     '#0f1320',
    'figure.facecolor':   '#0a0d14',
    'text.color':         '#e2e8f0',
    'axes.labelcolor':    '#8a96b0',
    'xtick.color':        '#8a96b0',
    'ytick.color':        '#8a96b0',
    'grid.color':         '#1e2840',
    'grid.linestyle':     '--',
    'grid.linewidth':     0.6,
})

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor('#0a0d14')
ax.set_facecolor('#0f1320')

BAR_COLORS  = ['#3b82f6', '#4fffb0', '#f472b6']
EDGE_COLORS = ['#60a5fa', '#6effc7', '#f9a8d4']

x = np.arange(len(names))
bars = ax.bar(
    x, values,
    width=0.42,
    color=BAR_COLORS,
    edgecolor=EDGE_COLORS,
    linewidth=1.4,
    zorder=3,
)

# Zoom Y axis so differences are clearly visible
y_min = max(0, min(values) - 0.015)
y_max = min(1.0, max(values) + 0.015)
ax.set_ylim(y_min, y_max)

ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

# Value labels on top of each bar
for bar, val, ec in zip(bars, values, EDGE_COLORS):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + (y_max - y_min) * 0.008,
        f'{val:.4f}',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold',
        color=ec,
    )

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=11, labelpad=10)
ax.set_title(
    'Accuracy vs Number of Hidden Layers',
    fontsize=14, fontweight='bold', color='#e2e8f0', pad=18
)

ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:.1%}'))

fig.text(
    0.99, 0.01,
    'Dataset: Sklearn Digits  |  Smit Bhavsar',
    ha='right', va='bottom', fontsize=7.5, color='#4a5568'
)

plt.tight_layout(pad=1.6)
plt.savefig('accuracy.png', dpi=180, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved -> accuracy.png')


In [8]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.facecolor':     '#0f1320',
    'figure.facecolor':   '#0a0d14',
    'text.color':         '#e2e8f0',
    'axes.labelcolor':    '#8a96b0',
    'xtick.color':        '#8a96b0',
    'ytick.color':        '#8a96b0',
    'grid.color':         '#1e2840',
    'grid.linestyle':     '--',
    'grid.linewidth':     0.6,
    'axes.spines.left':   True,
    'axes.spines.bottom': True,
})

model_names = list(histories.keys())
TRAIN_COL   = ['#3b82f6', '#4fffb0', '#f472b6']
VAL_COL     = ['#93c5fd', '#a7f3d0', '#fbcfe8']
PANEL_TITLE = ['\u2460 1 Hidden Layer', '\u2461 2 Hidden Layers', '\u2462 3 Hidden Layers']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.patch.set_facecolor('#0a0d14')
fig.suptitle(
    'Training vs Validation Accuracy \u2014 All Architectures',
    fontsize=14, fontweight='bold', color='#e2e8f0', y=1.02
)

for ax, name, tc, vc, ptitle in zip(axes, model_names, TRAIN_COL, VAL_COL, PANEL_TITLE):
    h      = histories[name].history
    train  = h['accuracy']
    val    = h['val_accuracy']
    epochs = range(1, len(train) + 1)

    ax.set_facecolor('#0f1320')

    # Shade overfitting gap
    ax.fill_between(epochs, train, val,
                    where=[t > v for t, v in zip(train, val)],
                    alpha=0.10, color=tc)

    ax.plot(epochs, train, color=tc, linewidth=2.2, label='Train')
    ax.plot(epochs, val,   color=vc, linewidth=2.2, linestyle='--', label='Validation')

    # Mark best validation epoch
    best_ep  = int(np.argmax(val)) + 1
    best_val = max(val)
    ax.scatter(best_ep, best_val, color=vc, s=60, zorder=5)
    ax.annotate(
        f'Best\n{best_val:.3f}',
        xy=(best_ep, best_val),
        xytext=(best_ep + 0.8, best_val - 0.03),
        fontsize=7.5, color=vc,
        arrowprops=dict(arrowstyle='->', color=vc, lw=0.9),
    )

    ax.yaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    ax.spines['left'].set_color('#1e2840')
    ax.spines['bottom'].set_color('#1e2840')

    ax.set_xlabel('Epoch', fontsize=10)
    if ax is axes[0]:
        ax.set_ylabel('Accuracy', fontsize=10)

    ax.set_title(ptitle, fontsize=11, fontweight='bold', color='#e2e8f0', pad=10)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.set_xlim(1, len(train))
    ax.set_ylim(0.3, 1.02)

    leg = ax.legend(fontsize=8.5, loc='lower right',
                    framealpha=0.15, edgecolor='#1e2840',
                    facecolor='#141927')
    for line in leg.get_lines():
        line.set_linewidth(2)

fig.text(
    0.99, -0.02,
    'Dataset: Sklearn Digits  |  Smit Bhavsar',
    ha='right', va='bottom', fontsize=7.5, color='#4a5568'
)

plt.tight_layout(pad=1.8)
plt.savefig('training_curve.png', dpi=180, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved -> training_curve.png')
